# Apache Spark Cheat Sheet
A complete reference from basic to advanced Spark usage in Python (PySpark).

In [ ]:

# ✅ Creating a Spark Session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark Cheat Sheet") \
    .master("local[*]") \
    .getOrCreate()


from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark Cheat Sheet") \
    .master("local[*]") \
    
    # ------------------ Hive Support ------------------
    .enableHiveSupport() \  # Needed if you use Hive tables
    
    # ------------------ Memory Configs ------------------
    .config("spark.executor.memory", "4g") \             # Memory per executor
    .config("spark.driver.memory", "2g") \               # Driver memory
    .config("spark.executor.memoryOverhead", "512m") \   # Extra overhead memory
    .config("spark.sql.shuffle.partitions", "200") \     # Number of partitions after shuffle
    .config("spark.default.parallelism", "100") \        # Default parallelism for RDDs

    # ------------------ Partition & File Handling ------------------
    .config("spark.sql.files.maxPartitionBytes", "128MB") \   # Size of each partition
    .config("spark.sql.files.openCostInBytes", "4MB") \       # Optimization for file listing
    .config("spark.sql.autoBroadcastJoinThreshold", "10MB") \ # Auto broadcast join threshold

    # ------------------ S3 / Hadoop AWS configs ------------------
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
    
    # ------------------ Performance & Optimization ------------------
    .config("spark.sql.adaptive.enabled", "true") \        # Enable Adaptive Query Execution
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \ 
    .config("spark.sql.broadcastTimeout", "300") \         # Broadcast timeout in seconds

    # ------------------ Warehousing ------------------
    .config("spark.sql.warehouse.dir", "/user/hive/warehouse") \

    # ------------------ Finalize Session ------------------
    .getOrCreate()


In [ ]:

# ✅ Creating DataFrame from list of tuples
data = [("Alice", 34), ("Bob", 45), ("Cathy", 29)]
columns = ["Name", "Age"]
df = spark.createDataFrame(data, columns)
df.show()




schema = StructType([
    StructField("Name", StringType(), nullable=False),  # Not nullable
    StructField("Age", IntegerType(), nullable=True)    # Nullable (default)
])

# Sample data
data = [("Alice", 34), ("Bob", None), ("Cathy", 29)]

# Create DataFrame with schema
df = spark.createDataFrame(data, schema=schema)



In [ ]:

# ✅ Reading CSV
df_csv = spark.read.csv("path/to/file.csv", header=True, inferSchema=True)
df_csv = spark.read.format("csv").option("header", True).option("inferSchema", True).load("path/to/file.csv")
# ✅ Reading JSON
df_json = spark.read.json("path/to/file.json")

# ✅ Reading Parquet
df_parquet = spark.read.parquet("path/to/file.parquet")

# ✅ Reading from HDFS
df_hdfs = spark.read.csv("hdfs://namenode:8020/path/to/file.csv", header=True)

# ✅ Reading from S3
df_s3 = spark.read.csv("s3a://your-bucket/path/to/file.csv", header=True)


In [ ]:

# ✅ Basic Transformations
df.select("Name").show()
df.withColumn("AgePlusTen", df["Age"] + 10).show()
df.filter(df["Age"] > 30).show()
df.groupBy("Age").count().show()


In [ ]:

# ✅ Joins
df1.join(df2, df1.id == df2.id, "inner")
df1.join(df2, df1.id == df2.id, "left")
df1.join(df2, df1.id == df2.id, "right")
df1.join(df2, df1.id == df2.id, "outer")


In [ ]:

from pyspark.sql.functions import broadcast

# ✅ Broadcast Join
df1.join(broadcast(df2), "id").show()


In [ ]:

# ✅ Accumulator Example
acc = spark.sparkContext.accumulator(0)

def add_to_acc(row):
    global acc
    acc += 1

df.foreach(add_to_acc)
print("Total rows processed:", acc.value)


In [ ]:

from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col

# ✅ Window Function Example
windowSpec = Window.partitionBy("department").orderBy(col("salary").desc())
df.withColumn("rank", rank().over(windowSpec)).show()


In [ ]:

# ✅ Using SQL with Spark
df.createOrReplaceTempView("people")
spark.sql("SELECT * FROM people WHERE Age > 30").show()


In [ ]:

# ✅ Spark Submit Examples

# Run on local
# spark-submit your_script.py

# Run on YARN cluster
# spark-submit --master yarn --deploy-mode cluster your_script.py

# With additional packages (e.g., S3 support)
# spark-submit --packages org.apache.hadoop:hadoop-aws:3.3.1 your_script.py


In [ ]:

# ✅ Stop Spark Session
spark.stop()


In [ ]:
spark-submit \
  --master local[*] \
  --py-files aggregation.zip \
  daily.py


In [ ]:
spark3-submit --deploy-mode cluster --name overall_metrics  --driver-memory 6G --executor-memory 4G --executor-cores 2 
--conf "spark.yarn.maxAppAttempts=1" --conf spark.kerberos.principal=jioapp@JIOENERGENIESIT.COM 
--conf spark.kerberos.keytab=/home/jioapp/jioapp.keytab --py-files /app/spark_jobs/git_project/Battery-Batch-Analytics-Engine/src.zip 
/app/spark_jobs/test_repo/sreenu/Battery-Batch-Analytics-Engine/src/aggregations/refactor_overall_aggregation.py bms 2025 2 1